# Activation Patching Variants

Advanced patching methods: denoising, noising, mean ablation, resample ablation, and causal mediation analysis. Fine-grained causal tools for model analysis.

**References:**
- Meng et al. (2022) "Locating and Editing Factual Associations in GPT"
- Vig et al. (2020) "Investigating Gender Bias Using Causal Mediation Analysis"

## Why This Matters

Different patching methods (denoising, noising, mean ablation, resample ablation) make different assumptions about what counts as a 'null' baseline. Understanding which variant to use and how results differ is essential for drawing valid causal conclusions from patching experiments.

**Key references:**
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.activation_patching_variants import (
    denoising_patching,
    noising_patching,
    mean_ablation,
    resample_ablation,
    causal_mediation_analysis,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

clean_tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
corrupted_tokens = jnp.array([40, 41, 42, 43, 44, 45, 46, 47])
def metric(logits): return float(logits[-1, 0])
print(f"Model: {cfg.n_layers} layers, {cfg.n_heads} heads")

In [ ]:
# 1. Denoising patching (run corrupted, restore from clean)
denoise = denoising_patching(model, corrupted_tokens, clean_tokens, metric)
print(f"Corrupted baseline: {denoise['baseline_metric']:.4f}")
print(f"Clean target: {denoise['clean_metric']:.4f}")
print(f"\nTop recovery fractions:")
sorted_recovery = sorted(denoise['recovery_fractions'].items(), key=lambda x: abs(x[1]), reverse=True)
for comp, frac in sorted_recovery[:5]:
    print(f"  {comp}: {frac:.4f}")

In [ ]:
# 2. Noising patching (run clean, corrupt each component)
noise = noising_patching(model, clean_tokens, corrupted_tokens, metric)
print(f"Clean baseline: {noise['baseline_metric']:.4f}")
print(f"Corrupted target: {noise['corrupted_metric']:.4f}")
print(f"\nTop damage fractions:")
sorted_damage = sorted(noise['damage_fractions'].items(), key=lambda x: abs(x[1]), reverse=True)
for comp, frac in sorted_damage[:5]:
    print(f"  {comp}: {frac:.4f}")

In [ ]:
# 3. Mean ablation (replace with mean activation)
mean_abl = mean_ablation(model, clean_tokens, metric, n_samples=3)
print(f"Baseline: {mean_abl['baseline_metric']:.4f}")
print(f"Most critical: {mean_abl['most_critical_component']}")
print(f"\nAttention effects:")
for l in range(cfg.n_layers):
    vals = [f"{mean_abl['attn_effects'][l, h]:.4f}" for h in range(cfg.n_heads)]
    print(f"  Layer {l}: {vals}")
print(f"MLP effects: {[f'{e:.4f}' for e in mean_abl['mlp_effects']]}")

In [ ]:
# 4. Resample ablation (average over multiple random corruptions)
resamp = resample_ablation(model, clean_tokens, metric, n_resamples=3)
print(f"Baseline: {resamp['baseline_metric']:.4f}")
print(f"\nMean effects (with std):")
for l in range(cfg.n_layers):
    for h in range(cfg.n_heads):
        eff = resamp['attn_effects'][l, h]
        std = resamp['attn_std'][l, h]
        if abs(eff) > 0.01:
            print(f"  L{l}H{h}: {eff:.4f} ± {std:.4f}")
    mlp_e = resamp['mlp_effects'][l]
    mlp_s = resamp['mlp_std'][l]
    if abs(mlp_e) > 0.01:
        print(f"  L{l}MLP: {mlp_e:.4f} ± {mlp_s:.4f}")

In [ ]:
# 5. Causal mediation analysis
for layer in range(cfg.n_layers):
    cma = causal_mediation_analysis(model, clean_tokens, corrupted_tokens, metric, mediator_layer=layer)
    print(f"Layer {layer}: total={cma['total_effect']:.4f}, "
          f"indirect={cma['indirect_effect']:.4f} ({cma['mediation_fraction']:.2%}), "
          f"attn={cma['attn_indirect']:.4f}, mlp={cma['mlp_indirect']:.4f}")